In [1]:
import os
import numpy as np
import h5py
from PIL import Image

# sharp image만 모아둔 디렉토리
# img_directory = f'/data1/ohjinjin/nas_ohjinjin/GoPro_sharp/test'
# img_directory = f'/data1/ohjinjin/GoPro_synthesis/test/'
flo_directory = f'/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin_sources/adjtemp/flow_flo_test/'
# EFNet에서 사용했던 h5 확장자의 GoPro with SCER dataset이 저장된 디렉토리
# h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_original/GOPRO/test'
# input h5 direcotry
h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan/test'
output_h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan_pred_of/test'
if not os.path.exists(output_h5_directory):
    os.makedirs(output_h5_directory)


def readFlow(fn):
    """ Read .flo file in Middlebury format"""
    # Code adapted from:
    # http://stackoverflow.com/questions/28013200/reading-middlebury-flow-files-with-python-bytes-array-numpy

    # WARNING: this will work on little-endian architectures (eg Intel x86) only!
    # print 'fn = %s'%(fn)
    with open(fn, 'rb') as f:
        magic = np.fromfile(f, np.float32, count=1)
        if 202021.25 != magic:
            print('Magic number incorrect. Invalid .flo file')
            return None
        else:
            w = np.fromfile(f, np.int32, count=1)
            h = np.fromfile(f, np.int32, count=1)
            # print 'Reading %d x %d flo file\n' % (w, h)
            data = np.fromfile(f, np.float32, count=2*int(w)*int(h))
            # Reshape data into 3D array (columns, rows, bands)
            # The reshape here is for visualization, the original code is (w,h,2)
            return np.resize(data, (int(h), int(w), 2))


# scene 별로 저장된 원본 .h5 file의 내용을 그대로 먼저 복사한 후 flow를 추가해준뒤 새 .h5로 저장해주는 함수
def process_files3(flo_directory, h5_directory):
    flo_files = [f for f in os.listdir(flo_directory) if f.endswith('.flo')]
    scene_dict = {}

    # scene별로 파일 분류
    for file in flo_files:
#         print(f'Curr file: {file}')
        scene = '_'.join(file.split('_')[:3])  # 예: 'GOPR0384_11_00'
        if scene not in scene_dict:
            scene_dict[scene] = []
        scene_dict[scene].append(file)
    
    
    for scene, filenames in scene_dict.items():
        print(f"Current scene is {scene}")
        
        filenames.sort()  # 원래의 순서대로 정렬
        
        old_h5_file_path = os.path.join(h5_directory, scene+".h5")
        new_h5_file_path = os.path.join(output_h5_directory, scene+".h5")
        with h5py.File(old_h5_file_path, 'r') as old_file:
            with h5py.File(new_h5_file_path, 'w') as new_file:
#                 print(old_file.keys())
                for group in old_file.keys():
                    old_file.copy(group, new_file)
#                     print(f'COPY GROUPs\n current group: {group}')
                if 'flows' not in new_file:
                    flows_group = new_file.create_group('flows')                    
                else:
                    flows_group = new_file['flows']
                for index, filename in enumerate(filenames, start=0):
                    flo_file_path = os.path.join(flo_directory, filename)
#                     print("flow_file: ", flo_file_path)
                    flow_data = np.transpose(readFlow(flo_file_path), (2,0,1))
                    flows_group.create_dataset(f'flow{str(index).zfill(9)}', data=flow_data)
        
        
#                 # scene마다 가장 처음과 가장 마지막 이미지는 사용할 수 없으므로 제외, EFNet에서 사용한 데이터도 그러함
#                 for index, filename in enumerate(filenames[1:-1], start=0):
#                     prev_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) - 1).zfill(6)+'.png')
#                     next_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) + 1).zfill(6)+'.png')

#                     # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
#                     prev_data = np.transpose(np.array(Image.open(prev_file_path))[:, :, ::-1], (2,0,1))
#                     next_data = np.transpose(np.array(Image.open(next_file_path))[:, :, ::-1], (2,0,1))

#                     prev_group.create_dataset(f'image{str(index).zfill(9)}', data=prev_data)
#                     next_group.create_dataset(f'image{str(index).zfill(9)}', data=next_data)

#     scene_dict = {}

#     # scene별로 sharp image 파일들 분류
#     for file in files:
# #         print(f'Curr file: {file}')
#         scene = '_'.join(file.split('_')[:3])  # 예: 'GOPR0384_11_00'
#         if scene not in scene_dict:
#             scene_dict[scene] = []
#         scene_dict[scene].append(file)

#     for scene, filenames in scene_dict.items():
#         print(f"Current scene is {scene}")
#         filenames.sort()

#         old_h5_file_path = os.path.join(h5_directory, scene+".h5")
#         new_h5_file_path = os.path.join(output_h5_directory, scene+".h5")
#         with h5py.File(old_h5_file_path, 'r') as old_file:
#             with h5py.File(new_h5_file_path, 'w') as new_file:
#                 for group in old_file.keys():
# #                     print(f'COPY GROUPs\n current group: {group}')
#                     old_file.copy(group, new_file)
# #                 print("Contents:", list(new_file.keys()))
#                 if 'sharp_images_prev' not in new_file:
#                     prev_group = new_file.create_group('sharp_images_prev')
#                 else:
#                     prev_group = new_file['sharp_images_prev']
                    
#                 if 'sharp_images_next' not in new_file:
#                     next_group = new_file.create_group('sharp_images_next')
#                 else:
#                     next_group = new_file['sharp_images_next']
                
#                 # scene마다 가장 처음과 가장 마지막 이미지는 사용할 수 없으므로 제외, EFNet에서 사용한 데이터도 그러함
#                 for index, filename in enumerate(filenames[1:-1], start=0):
#                     prev_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) - 1).zfill(6)+'.png')
#                     next_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) + 1).zfill(6)+'.png')

#                     # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
#                     prev_data = np.transpose(np.array(Image.open(prev_file_path))[:, :, ::-1], (2,0,1))
#                     next_data = np.transpose(np.array(Image.open(next_file_path))[:, :, ::-1], (2,0,1))

#                     prev_group.create_dataset(f'image{str(index).zfill(9)}', data=prev_data)
#                     next_group.create_dataset(f'image{str(index).zfill(9)}', data=next_data)

                    
# 파일 처리 시작
process_files3(flo_directory, h5_directory)


Current scene is GOPR0410_11_00
Current scene is GOPR0384_11_00
Current scene is GOPR0868_11_00
Current scene is GOPR0871_11_00
Current scene is GOPR0396_11_00
Current scene is GOPR0384_11_05
Current scene is GOPR0881_11_01
Current scene is GOPR0862_11_00
Current scene is GOPR0385_11_01
Current scene is GOPR0854_11_00
Current scene is GOPR0869_11_00


In [2]:
import os
import h5py
import numpy as np

with h5py.File('/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan_pred_of/test/GOPR0410_11_00.h5', 'r') as file:
    # 여기에서 파일 내 데이터를 읽고 처리할 수 있습니다.
    # 예를 들어, 모든 키를 출력할 수 있습니다.
    print("Contents:", list(file.keys()))
    print("c0:", list(file['images'].keys()))
    print("c0:", np.array(file['images/image000000001']).shape)
    print("c1:", list(file['masks'].keys()))
    print("c1:", np.array(file['masks/mask000000001']).shape)
    print("c2:", list(file['sharp_images'].keys()))
    print("c2:", np.array(file['sharp_images/image000000001']).shape)
    print("c3:", list(file['voxels'].keys()))
    print("c3:", np.array(file['voxels/voxel000000001']).shape)
    print("c4:", list(file['flows'].keys()))
    print("c4:", np.array(file['flows/flow000000001']).shape)

    
    
    
    # 특정 데이터셋을 읽으려면 file['dataset_name']을 사용하

Contents: ['flows', 'images', 'masks', 'sharp_images', 'sharp_images_next', 'sharp_images_prev', 'synthesized_images', 'voxels']
c0: ['image000000000', 'image000000001', 'image000000002', 'image000000003', 'image000000004', 'image000000005', 'image000000006', 'image000000007', 'image000000008', 'image000000009', 'image000000010', 'image000000011', 'image000000012', 'image000000013', 'image000000014', 'image000000015', 'image000000016', 'image000000017', 'image000000018', 'image000000019', 'image000000020', 'image000000021', 'image000000022', 'image000000023', 'image000000024', 'image000000025', 'image000000026', 'image000000027', 'image000000028', 'image000000029', 'image000000030', 'image000000031', 'image000000032', 'image000000033', 'image000000034', 'image000000035', 'image000000036', 'image000000037', 'image000000038', 'image000000039', 'image000000040', 'image000000041', 'image000000042', 'image000000043', 'image000000044', 'image000000045', 'image000000046', 'image000000047', 'i

c3: (6, 720, 1280)
c4: ['flow000000000', 'flow000000001', 'flow000000002', 'flow000000003', 'flow000000004', 'flow000000005', 'flow000000006', 'flow000000007', 'flow000000008', 'flow000000009', 'flow000000010', 'flow000000011', 'flow000000012', 'flow000000013', 'flow000000014', 'flow000000015', 'flow000000016', 'flow000000017', 'flow000000018', 'flow000000019', 'flow000000020', 'flow000000021', 'flow000000022', 'flow000000023', 'flow000000024', 'flow000000025', 'flow000000026', 'flow000000027', 'flow000000028', 'flow000000029', 'flow000000030', 'flow000000031', 'flow000000032', 'flow000000033', 'flow000000034', 'flow000000035', 'flow000000036', 'flow000000037', 'flow000000038', 'flow000000039', 'flow000000040', 'flow000000041', 'flow000000042', 'flow000000043', 'flow000000044', 'flow000000045', 'flow000000046', 'flow000000047', 'flow000000048', 'flow000000049', 'flow000000050', 'flow000000051', 'flow000000052', 'flow000000053', 'flow000000054', 'flow000000055', 'flow000000056', 'flow00

In [ ]:
# # 잘 저장되었는지 확인하기 위한 셀
# import matplotlib.pyplot as plt
# # import h5py
# # import numpy as np
# with h5py.File('/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan/test/GOPR0384_11_00.h5', 'r') as file:
#     print("Contents:", list(file.keys()))
# #     print("c0:", list(file['images'].keys()))
# #     print("c0:", np.array(file['images/image000000000']))
# #     print("c1:", list(file['masks'].keys()))
# #     print("c1:", np.array(file['masks/mask000000001']).shape)
#     print("c1:", list(file['images'].keys()))
#     print("c1:", list(file['synthesized_images'].keys()))
#     print("c1:", list(file['sharp_images'].keys()))
#     print("c2:", list(file['sharp_images_prev'].keys()))
#     print("c2:", list(file['sharp_images_next'].keys()))
    
# #     print("c2:", np.array(file['sharp_images/image000000001']).shape)
# #     print("c3:", list(file['voxels'].keys()))
# #     print("c3:", np.array(file['voxels/voxel000000001']).shape)
# #     print("c4:", list(file['flows'].keys()))
# #     print("c4:", np.array(file['flows/flow000000001']).shape)
#     ori_blur_img = np.array(file['images/image000000000'])
#     syn_blur_img = np.array(file['synthesized_images/image000000000'])
#     img = np.array(file['sharp_images/image000000000'])
#     imgprev = np.array(file['sharp_images_prev/image000000000'])
#     imgnext = np.array(file['sharp_images_next/image000000000'])
#     plt.title("ori_blur_img")
#     plt.imshow(np.transpose(ori_blur_img,(1,2,0))[:, :, [2, 1, 0]])
#     plt.show()
#     plt.title("syn_blur_img")
#     plt.imshow(np.transpose(syn_blur_img,(1,2,0))[:, :, [2, 1, 0]])
#     plt.show()
#     plt.title("sharp_img")
#     plt.imshow(np.transpose(img,(1,2,0))[:, :, [2, 1, 0]])
#     plt.show()
#     plt.title("imgprev")
#     plt.imshow(np.transpose(imgprev,(1,2,0))[:, :, [2, 1, 0]])
#     plt.show()
#     plt.title("imgnext")
#     plt.imshow(np.transpose(imgnext,(1,2,0))[:, :, [2, 1, 0]])
#     plt.show()
    
    

In [3]:
flo_directory = f'/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin_sources/adjtemp/flow_flo_train/'
# EFNet에서 사용했던 h5 확장자의 GoPro with SCER dataset이 저장된 디렉토리
# h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_original/GOPRO/test'
# input h5 direcotry
h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan/train'
output_h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan_pred_of/train'
if not os.path.exists(output_h5_directory):
    os.makedirs(output_h5_directory)
    
# 파일 처리 시작
process_files3(flo_directory, h5_directory)

Current scene is GOPR0868_11_01
Current scene is GOPR0385_11_00
Current scene is GOPR0868_11_02
Current scene is GOPR0871_11_01
Current scene is GOPR0380_11_00
Current scene is GOPR0374_11_01
Current scene is GOPR0374_11_00
Current scene is GOPR0857_11_00
Current scene is GOPR0386_11_00
Current scene is GOPR0379_11_00
Current scene is GOPR0378_13_00
Current scene is GOPR0374_11_02
Current scene is GOPR0372_07_01
Current scene is GOPR0384_11_01
Current scene is GOPR0884_11_00
Current scene is GOPR0881_11_00
Current scene is GOPR0384_11_04
Current scene is GOPR0477_11_00
Current scene is GOPR0384_11_03
Current scene is GOPR0372_07_00
Current scene is GOPR0384_11_02
Current scene is GOPR0374_11_03
